In [0]:
%sql
-- ============================================================
-- ETL: silver.ventas → gold.fact_ventas
-- Patrón: MERGE (idempotente — se puede correr N veces)
-- Granularidad: 1 fila = 1 línea de venta
-- pedido_key: clave degenerada para reconstruir pedidos multi-item
-- total_venta: calculado acá, no en Silver
-- ============================================================

MERGE INTO iniciacion_deportiva.gold.fact_ventas AS target
USING (

    WITH

    -- =====================================================
    -- PASO 1: Traer ventas desde Silver con FKs resueltas
    -- JOIN a dim_producto (is_current = TRUE)
    -- JOIN a dim_tiempo por fecha casteada a INT YYYYMMDD
    -- =====================================================
    ventas_con_keys AS (
        SELECT
            v.id_venta,
            v.id_cliente,
            v.precio_unitario,
            v.cantidad,
            v.comision,
            v.precio_unitario * v.cantidad          AS total_venta,
            v.estado,
            v.fecha,
            -- FK dim_producto — título activo actual
            dp.producto_key,
            -- FK dim_tiempo — fecha como INT YYYYMMDD
            CAST(DATE_FORMAT(CAST(v.fecha AS DATE), 'yyyyMMdd') AS INT) AS tiempo_key,
            -- Clave degenerada de pedido
            -- Agrupa líneas del mismo cliente en el mismo minuto
            CONCAT(
                v.id_cliente, '_',
                DATE_FORMAT(v.fecha, 'yyyyMMddHHmm')
            )                                       AS pedido_key
        FROM iniciacion_deportiva.silver.ventas v
        -- JOIN a dim_producto: solo fila activa
        LEFT JOIN iniciacion_deportiva.gold.dim_producto dp
            ON v.id_producto = dp.id_producto
            AND dp.es_actual = TRUE
    )

    SELECT * FROM ventas_con_keys

) AS source
ON target.id_venta = source.id_venta

-- Si la venta existe y el estado cambió → actualizamos
WHEN MATCHED AND target.estado != source.estado
    THEN UPDATE SET
        target.estado                   = source.estado,
        target._processing_timestamp    = CURRENT_TIMESTAMP()

-- Si la venta no existe → insertamos
WHEN NOT MATCHED
    THEN INSERT (
        producto_key,
        tiempo_key,
        pedido_key,
        id_venta,
        id_cliente,
        precio_unitario,
        cantidad,
        comision,
        total_venta,
        estado,
        _source,
        _processing_timestamp
    )
    VALUES (
        source.producto_key,
        source.tiempo_key,
        source.pedido_key,
        source.id_venta,
        source.id_cliente,
        source.precio_unitario,
        source.cantidad,
        source.comision,
        source.total_venta,
        source.estado,
        'iniciacion_deportiva.silver.ventas',
        CURRENT_TIMESTAMP()
    );